# TTS Pipeline — Colab GPU Worker

This notebook runs the Abogen TTS engine + pipeline worker on Colab's free GPU.

## Before you start

**On your local machine**, expose Redis so Colab can reach it:
```bash
# Option A: ngrok (recommended)
ngrok tcp 6379
# → copy the  tcp://X.tcp.ngrok.io:XXXXX  URL

# Option B: cloudflared
cloudflared tunnel --url tcp://localhost:6379
```

Then fill in `REDIS_URL` and `WORKER_ID` in the Config cell below.

**MP3 output** goes to Google Drive → your local merger reads it from the same Drive mount.

In [ ]:
# ── 1. Mount Google Drive (MP3 output) ────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
OUTPUT_DIR = '/content/drive/MyDrive/tts-pipeline/outputs'
SPOOL_DIR  = '/content/spool'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(SPOOL_DIR,  exist_ok=True)
print('Drive mounted. Output dir:', OUTPUT_DIR)

In [ ]:
# ── 2. Secrets ────────────────────────────────────────────────────────────────
# Add these in the Secrets panel (🔑 icon, left sidebar):
#
#   GITHUB_TOKEN      → ghp_...  (Personal Access Token — private repos only)
#   GITHUB_USER       → your GitHub username
#   NGROK_TOKEN       → ngrok authtoken (dashboard.ngrok.com → Your Authtoken)
#
#   SSH_HOST          → server IP or hostname  e.g. 10.0.0.1
#   SSH_USER          → SSH username           e.g. root
#   SSH_KEY           → private key contents   (paste the full -----BEGIN ... KEY-----)
#   SSH_REMOTE_PATH   → remote output dir      e.g. /opt/tts-node/outputs
#   SSH_PORT          → SSH port (optional, default 22)
#
from google.colab import userdata

GITHUB_TOKEN    = userdata.get('GITHUB_TOKEN')
GITHUB_USER     = userdata.get('GITHUB_USER')
NGROK_TOKEN     = userdata.get('NGROK_TOKEN')
SSH_HOST        = userdata.get('SSH_HOST')
SSH_USER        = userdata.get('SSH_USER')
SSH_KEY         = userdata.get('SSH_KEY')
SSH_REMOTE_PATH = userdata.get('SSH_REMOTE_PATH') or '/opt/tts-node/outputs'
SSH_PORT        = int(userdata.get('SSH_PORT') or 22)

if GITHUB_TOKEN and GITHUB_USER:
    import subprocess
    subprocess.run([
        'git', 'config', '--global', 'credential.helper',
        f'!f() {{ echo username={GITHUB_USER}; echo password={GITHUB_TOKEN}; }}; f'
    ], check=True)
    print(f'Git auth: {GITHUB_USER}')
else:
    print('Git auth: skipped (public repo)')

print(f'ngrok token:  {"ok" if NGROK_TOKEN     else "MISSING"}')
print(f'SSH host:     {"ok" if SSH_HOST        else "MISSING"}')
print(f'SSH key:      {"ok" if SSH_KEY         else "MISSING"}')

In [ ]:
# ── SSH setup + mount remote output dir ──────────────────────────────────────
import os, subprocess, textwrap, re

if not (SSH_HOST and SSH_USER and SSH_KEY):
    raise RuntimeError('SSH_HOST, SSH_USER and SSH_KEY must all be set in Secrets.')

SSH_KEY_PATH = '/root/.ssh/colab_key'
os.makedirs('/root/.ssh', exist_ok=True)

def _reconstruct_pem(raw: str) -> str:
    # 1. Strip all leading/trailing whitespace and normalize literal '\n'
    key = raw.replace('\\n', '\n').strip()

    # 2. Extract header, body, and footer regardless of spacing
    pattern = r'(-----BEGIN [^-]+-----)(.*?)(-----END [^-]+-----)'
    match = re.search(pattern, key, re.DOTALL)
    if match:
        header, body, footer = match.groups()
        # Clean body: remove ALL whitespace/newlines from the base64 part
        body = "".join(body.split())
        # Wrap base64 body at 64 chars
        wrapped_body = '\n'.join(body[i:i+64] for i in range(0, len(body), 64))
        return f'{header}\n{wrapped_body}\n{footer}\n'

    return key + '\n'

with open(SSH_KEY_PATH, 'w') as f:
    f.write(_reconstruct_pem(SSH_KEY))
os.chmod(SSH_KEY_PATH, 0o600)

# Validate key format
check = subprocess.run(['ssh-keygen', '-y', '-f', SSH_KEY_PATH], capture_output=True, text=True)
if check.returncode != 0:
    print("Debug - First 100 chars of processed key:\n", open(SSH_KEY_PATH).read()[:100])
    raise RuntimeError(f'SSH Key Error: {check.stderr}')

# Configure SSH client
with open('/root/.ssh/config', 'w') as f:
    f.write(textwrap.dedent(f"""\
        Host {SSH_HOST}
            StrictHostKeyChecking no
            UserKnownHostsFile /dev/null
            IdentityFile {SSH_KEY_PATH}
            Port {SSH_PORT}
            ConnectTimeout 10
    """))

print(f'SSH key verified. Testing connection to {SSH_USER}@{SSH_HOST}...')

# Test connectivity and ensure remote path exists
test_conn = subprocess.run(['ssh', f'{SSH_USER}@{SSH_HOST}', f'mkdir -p {SSH_REMOTE_PATH}'], capture_output=True, text=True)
if test_conn.returncode != 0:
    raise RuntimeError(f'Connection failed: {test_conn.stderr}')

# Mount via sshfs
!apt-get install -y sshfs > /dev/null 2>&1
SSHFS_MOUNT = '/content/remote-outputs'
os.makedirs(SSHFS_MOUNT, exist_ok=True)
subprocess.run(['fusermount', '-uz', SSHFS_MOUNT], capture_output=True)

mount_cmd = [
    'sshfs', f'{SSH_USER}@{SSH_HOST}:{SSH_REMOTE_PATH}', SSHFS_MOUNT,
    '-o', f'IdentityFile={SSH_KEY_PATH},port={SSH_PORT}',
    '-o', 'reconnect,ServerAliveInterval=15,ServerAliveCountMax=3'
]

result = subprocess.run(mount_cmd, capture_output=True, text=True)
if result.returncode == 0:
    OUTPUT_DIR = SSHFS_MOUNT
    os.environ['OUTPUT_DIR'] = OUTPUT_DIR
    print(f'Successfully mounted {SSH_REMOTE_PATH} to {SSHFS_MOUNT}')
else:
    print(f'Mount failed: {result.stderr}')

In [ ]:
# ── 3. Config — fill these in ─────────────────────────────────────────────────
REDIS_URL  = 'rediss://redis.0xvictor.dev:6380'
WORKER_ID  = 'colab-gpu-1'

# ABOGEN_HOST here is the URL the worker uses to call the local Abogen API.
# Note: Abogen's app.py uses ABOGEN_HOST as its bind address (0.0.0.0),
# so we keep that separate — see cell 6.
os.environ['REDIS_URL']       = REDIS_URL
os.environ['WORKER_ID']       = WORKER_ID
os.environ['ABOGEN_HOST']     = 'http://localhost:8808'
os.environ['OUTPUT_DIR']      = OUTPUT_DIR
os.environ['SPOOL_DIR']       = SPOOL_DIR
os.environ['USE_GPU']         = 'true'
os.environ['PROMETHEUS_PORT'] = '8000'
print('Config set.')

In [ ]:
# ── 4. Install system deps ────────────────────────────────────────────────────
!apt-get install -y espeak-ng ffmpeg > /dev/null 2>&1
print('espeak-ng + ffmpeg installed.')

In [ ]:
# ── 5. Install Abogen + GPU PyTorch ──────────────────────────────────────────
!pip install -q torch torchaudio --index-url https://download.pytorch.org/whl/cu121
!git clone --depth=1 https://github.com/wizardgang/audiobook-stack /content/audiobook-stack 2>/dev/null || \
 git -C /content/audiobook-stack pull
!pip install -q -e '/content/audiobook-stack/abogen_src[gpu-cu121]' 2>/dev/null || \
 pip install -q -e '/content/audiobook-stack/abogen_src'
!pip install -q redis prometheus_client requests pyngrok
print('All packages installed.')

In [ ]:
# ── 7. Verify Redis connection ────────────────────────────────────────────────
import redis
r = redis.from_url(REDIS_URL, decode_responses=True, socket_connect_timeout=5)
print('Redis ping:', r.ping())
print('TTS queue depth:', r.llen('pipeline:tts'))

In [ ]:
# # ── TEMP: Push existing Drive output → server via rsync ──────────────────────
# # Run this cell standalone to sync whatever is already in Google Drive to the server.
# # Requires: Drive mounted (cell 1) + SSH secrets loaded (cell 2).
# # Safe to re-run — rsync skips files already on the server.

# import subprocess, os

# DRIVE_OUTPUT = '/content/drive/MyDrive/tts-pipeline/outputs'  # adjust if needed

# if not os.path.isdir(DRIVE_OUTPUT):
#     raise RuntimeError(f'Drive path not found: {DRIVE_OUTPUT} — run cell 1 first.')
# if not (SSH_HOST and SSH_USER and SSH_KEY):
#     raise RuntimeError('SSH secrets not loaded — run cell 2 first.')

# # Count what we're about to push
# mp3_count = sum(1 for _, _, files in os.walk(DRIVE_OUTPUT) for f in files if f.endswith('.mp3'))
# print(f'Found {mp3_count} MP3 files in {DRIVE_OUTPUT}')
# print(f'Pushing to {SSH_USER}@{SSH_HOST}:{SSH_REMOTE_PATH} ...')

# result = subprocess.run([
#     'rsync', '-avz', '--progress',
#     '-e', f'ssh -i {SSH_KEY_PATH} -p {SSH_PORT} -o StrictHostKeyChecking=no',
#     DRIVE_OUTPUT + '/',          # trailing slash = contents only, not the dir itself
#     f'{SSH_USER}@{SSH_HOST}:{SSH_REMOTE_PATH}/',
# ], capture_output=False, text=True)   # capture_output=False so progress prints live

# if result.returncode != 0:
#     print('rsync failed — check output above.')
# else:
#     print(f'\nDone. {mp3_count} files synced to server.')